# 보안 취약점 일일 브리핑 자동화

[agent_suggest.md](agent_suggest.md) 의 **1번 사례**를 실제로 구현합니다.

## 해결하려는 문제

매일 수십~수백 건의 신규 CVE가 공개되지만, 그중 **우리 회사 자산에 해당하는 것은 극소수**입니다.
담당자는 이걸 걸러내느라 매일 아침 1~2시간을 씁니다.

## 전체 흐름

```
[1] 자산 목록 로드          assets.json (가상의 자사 자산 10건)
        ↓
[2] CVE 수집 (도구)         NVD 공개 API — 제품별 최근 취약점
        ↓
[3] 자산 매칭 에이전트      키워드만 겹치는 오탐 제거 + 버전 판단
        ↓
[4] 위험도 평가 에이전트    CVSS + 자산 중요도 + 인터넷 노출 → 등급/기한/조치
        ↓
[5] 담당자별 메일 발송      발송 에이전트가 메일 도구를 담당자 수만큼 호출 (팬아웃)
```

## 설계 원칙

| 단계 | 담당 | 이유 |
|---|---|---|
| 자산 로드, CVE 조회 | **파이썬 코드** | 정해진 규칙대로 가져오면 되는 일. LLM이 할 이유가 없음 |
| 오탐 제거, 위험도 판단 | **LLM 에이전트** | "이 CVE가 우리 것인가"는 규칙으로 못 짬 |
| 메일 발송 | **도구(코드) + 발송 에이전트** | 외부로 나가는 행동. 에이전트는 받을 담당자만 고르고, 주소·본문·중복 방지는 도구 코드가 통제 |

**판단은 모델에게, 계산과 조회는 코드에게** — 이것이 이 노트북의 핵심입니다.

In [1]:
import os
import json
import time
import smtplib
import requests
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool, ModelSettings

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
ASSETS_PATH = "assets.json"

# NVD API 키 (.env 의 NVD_KEY). 없어도 동작하지만 호출 제한이 10배 빡빡해집니다.
NVD_KEY = os.getenv("NVD_KEY")

# 호출 간격: 키가 있으면 30초당 50회, 없으면 30초당 5회
NVD_DELAY = 0.6 if NVD_KEY else 6.0

print(f"NVD API 키: {'사용' if NVD_KEY else '없음 (익명 호출)'} / 호출 간격 {NVD_DELAY}초")

NVD API 키: 사용 / 호출 간격 0.6초


## 1단계. 자산 목록

실무에서는 CMDB(자산관리시스템)나 자산 인벤토리에서 가져옵니다.
여기서는 가상의 회사 자산 10건을 `assets.json` 에 넣어 두었습니다.

각 자산에는 취약점 판단에 꼭 필요한 네 가지가 들어 있습니다.

- **product / version** — 어떤 CVE가 해당되는지 판단하는 기준
- **owner** — 누구에게 메일을 보낼지
- **criticality** — 자산 중요도 (상/중/하)
- **internet_facing** — 인터넷 노출 여부. **같은 CVE라도 노출된 자산이면 더 위험합니다**

In [3]:
def load_assets():
    """자산 인벤토리를 읽어옵니다."""
    with open(ASSETS_PATH, encoding="utf-8") as f:
        return json.load(f)


data = load_assets()
assets = data["assets"]

print(f"{data['company']} — 자산 {len(assets)}건\n")
print(f"{'ID':<12} {'제품':<22} {'버전':<10} {'담당자':<8} {'중요도':<6} 인터넷노출")
print("-" * 78)
for a in assets:
    print(f"{a['id']:<12} {a['product']:<22} {a['version']:<10} "
          f"{a['owner']:<8} {a['criticality']:<6} {'예' if a['internet_facing'] else '아니오'}")

(주)누리시스템 — 자산 10건

ID           제품                     버전         담당자      중요도    인터넷노출
------------------------------------------------------------------------------
web-nginx-01 nginx                  1.24.0     김인프라     상      예
web-ssl-02   OpenSSL                3.0.11     김인프라     상      예
was-tc-03    Apache Tomcat          9.0.71     김인프라     상      예
api-node-04  Node.js                18.17.1    박백엔드     상      예
cache-rd-05  Redis                  7.0.12     박백엔드     중      아니오
db-pg-06     PostgreSQL             15.4       이디비      상      아니오
ci-jk-07     Jenkins                2.414.1    최데브옵스    중      아니오
scm-gl-08    GitLab                 16.3.1     최데브옵스    상      아니오
mon-graf-09  Grafana                10.1.2     최데브옵스    하      예
legacy-st-10 Apache Struts          2.5.30     정레거시     중      아니오


## 2단계. CVE 수집 도구 (NVD API)

미국 NIST가 운영하는 **NVD(National Vulnerability Database)** 공개 API를 사용합니다.

### API 키

키 없이도 쓸 수 있지만, **호출 제한이 10배 차이 납니다.**

| 구분 | 호출 제한 | 호출 간격 | 제품 10종 조회 (실측) |
|---|---|---|---|
| 키 없음 | 30초당 **5회** | 6초 | 약 85초 |
| 키 있음 | 30초당 **50회** | 0.6초 | 약 30초 |

키는 [nvd.nist.gov/developers/request-an-api-key](https://nvd.nist.gov/developers/request-an-api-key)
에서 무료로 발급받아 `.env` 에 `NVD_KEY=...` 로 넣습니다.
요청 시에는 **`apiKey` 헤더**로 보냅니다 (쿼리 파라미터가 아닙니다).

> 키가 있어도 30초가 걸리는 이유는 **NVD 서버 응답 자체가 요청당 약 2.5초**이기 때문입니다.
> 호출 간격을 더 줄여도 이 시간은 줄지 않습니다.

[openai_agent_intro.ipynb](openai_agent_intro.ipynb) 의 날씨 도구와 같은 방식으로,
**에이전트에 붙이기 전에 함수부터 직접 호출해 검증**합니다.

In [4]:
def search_cves_for_product(product: str, days: int = 120, max_results: int = 4):
    """특정 제품과 관련된 최근 CVE를 NVD에서 조회합니다.

    Args:
        product: 제품 이름 (예: Apache Tomcat)
        days: 최근 며칠 이내에 공개된 것을 볼지
        max_results: 제품당 최대 건수
    """
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)

    # 키가 있으면 apiKey 헤더로 보낸다. 없으면 익명 호출(제한이 더 빡빡함).
    headers = {"apiKey": NVD_KEY} if NVD_KEY else {}

    r = requests.get(
        "https://services.nvd.nist.gov/rest/json/cves/2.0",
        params={
            "keywordSearch": product,
            "pubStartDate": start.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "pubEndDate": end.strftime("%Y-%m-%dT%H:%M:%S.000"),
            "resultsPerPage": max_results,
            "noRejected": None,
        },
        headers=headers,
        timeout=30,
    )
    r.raise_for_status()

    result = []
    for item in r.json().get("vulnerabilities", []):
        cve = item["cve"]
        metrics = cve.get("metrics", {})
        # CVSS 버전이 여러 개일 수 있으므로 최신 것부터 찾는다
        key = next((k for k in ["cvssMetricV31", "cvssMetricV40", "cvssMetricV30", "cvssMetricV2"]
                    if k in metrics), None)
        cvss = metrics[key][0]["cvssData"] if key else {}

        result.append({
            "cve_id": cve["id"],
            "published": cve["published"][:10],
            "cvss_score": float(cvss.get("baseScore", 0.0)),
            "severity": cvss.get("baseSeverity", "UNKNOWN"),
            "description": cve["descriptions"][0]["value"][:400],
        })
    return result


# 함수만 먼저 테스트
for cve in search_cves_for_product("Apache Tomcat", max_results=3):
    print(f"{cve['cve_id']}  CVSS {cve['cvss_score']:<5} {cve['severity']:<9} {cve['description'][:70]}")

CVE-2026-50229  CVSS 6.1   MEDIUM    Improper Neutralization of Script-Related HTML Tags in a Web Page (Bas
CVE-2026-53404  CVSS 7.3   HIGH      Always-Incorrect Control Flow Implementation vulnerability in Apache T
CVE-2026-53434  CVSS 9.1   CRITICAL  Detection of Error Condition Without Action vulnerability in Apache To


### 전체 제품 조회

이제 자산에 있는 모든 제품을 조회합니다.
`NVD_DELAY` 에 따라 **키가 있으면 약 6초, 없으면 약 1분** 걸립니다.

In [5]:
products = sorted({a["product"] for a in assets})
estimated = int(len(products) * NVD_DELAY)
print(f"제품 {len(products)}종 조회 시작 (약 {estimated}초)\n")

candidates = {}
for i, product in enumerate(products):
    candidates[product] = search_cves_for_product(product)
    print(f"  [{i + 1}/{len(products)}] {product:<22} -> {len(candidates[product])}건")
    if i < len(products) - 1:
        time.sleep(NVD_DELAY)   # NVD 호출 제한 회피 (키 유무에 따라 0.6초 / 6초)

total = sum(len(v) for v in candidates.values())
print(f"\nCVE 후보 총 {total}건 수집 완료")

제품 10종 조회 시작 (약 6초)

  [1/10] Apache Struts          -> 4건
  [2/10] Apache Tomcat          -> 4건
  [3/10] GitLab                 -> 4건
  [4/10] Grafana                -> 4건
  [5/10] Jenkins                -> 4건
  [6/10] Node.js                -> 4건
  [7/10] OpenSSL                -> 4건
  [8/10] PostgreSQL             -> 4건
  [9/10] Redis                  -> 4건
  [10/10] nginx                  -> 4건

CVE 후보 총 40건 수집 완료


## 3단계. 자산 매칭 에이전트

여기서부터가 **에이전트가 필요한 이유**입니다.

키워드 검색 결과에는 **이름만 겹치는 다른 제품**이 잔뜩 섞여 있습니다.

- 자산은 `OpenSSL` 인데 → CVE는 `rust-openssl`(Rust 바인딩) 취약점
- 자산은 `Node.js` 인데 → CVE는 특정 npm 패키지 취약점

이걸 정규식이나 문자열 매칭으로 거르는 것은 사실상 불가능합니다.
**설명을 읽고 판단해야** 합니다.

### 구조화 출력 (`output_type`)

에이전트가 자유 문장으로 답하면 다음 단계에서 파싱하기 어렵습니다.
Pydantic 모델을 `output_type` 으로 주면 **결과가 항상 정해진 구조로 나옵니다.**

In [6]:
class MatchedCVE(BaseModel):
    cve_id: str
    asset_id: str
    reason: str          # 왜 영향을 준다고 판단했는지
    cvss_score: float
    summary_ko: str      # 취약점 내용 한글 요약


class ExcludedCVE(BaseModel):
    cve_id: str
    reason: str          # 왜 제외했는지


class MatchResult(BaseModel):
    matched: list[MatchedCVE]
    excluded: list[ExcludedCVE]


matcher = Agent(
    name="자산 매칭 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 취약점 관리 담당자입니다. 자산 목록과 CVE 후보 목록을 받습니다.\n"
        "각 CVE가 해당 자산에 '실제로' 영향을 주는지 판단하세요.\n"
        "- 키워드만 겹치고 다른 제품이면 제외하세요. "
        "(예: 자산은 OpenSSL인데 CVE는 rust-openssl 바인딩 취약점)\n"
        "- 설명에 명시된 영향 버전 범위가 자산 버전을 포함하지 않으면 제외하세요.\n"
        "- 버전 정보가 불명확하면 matched 에 넣되 reason에 '버전 확인 필요'라고 적으세요.\n"
        "제외한 것은 excluded 에 이유와 함께 넣으세요. reason은 한 문장으로 쓰세요."
    ),
    output_type=MatchResult,
)

In [7]:
payload = {
    "assets": [
        {k: a[k] for k in ("id", "name", "product", "version", "criticality", "internet_facing")}
        for a in assets
    ],
    "cve_candidates": candidates,
}

with trace("보안 브리핑 - 자산 매칭"):
    m = await Runner.run(matcher, json.dumps(payload, ensure_ascii=False))

matched = m.final_output.matched
print(f"후보 {total}건 -> 실제 영향 {len(matched)}건 (제외 {len(m.final_output.excluded)}건)\n")

print("[ 영향받는 취약점 ]")
for x in matched:
    print(f"  {x.cve_id}  CVSS {x.cvss_score:<5} {x.asset_id:<12} {x.summary_ko[:55]}")

print("\n[ 제외된 예시 — 에이전트가 걸러낸 오탐 ]")
for x in m.final_output.excluded[:5]:
    print(f"  {x.cve_id}: {x.reason[:80]}")

후보 40건 -> 실제 영향 11건 (제외 27건)

[ 영향받는 취약점 ]
  CVE-2026-50229  CVSS 6.1   was-tc-03    Tomcat number guess 예제의 XSS 취약점으로, 9.0.71 버전이 영향 범위에 포함
  CVE-2026-53404  CVSS 7.3   was-tc-03    Tomcat rewrite valve의 제어 흐름 오류로, 9.0.71 버전이 영향 범위에 포함됩니
  CVE-2026-55276  CVSS 9.1   was-tc-03    Tomcat 웹.xml 로깅 관련 결함으로, 9.0.71 버전이 영향 범위에 포함됩니다.
  CVE-2026-10086  CVSS 8.7   scm-gl-08    GitLab EE의 클라이언트 측 코드 실행 취약점은 16.4 이상에서 영향이 확인되어 16.3.1
  CVE-2026-10712  CVSS 8.0   scm-gl-08    GitLab CE/EE의 JavaScript 실행 취약점은 18.10 이상 기준이라 16.3.1은 
  CVE-2026-33382  CVSS 7.5   mon-graf-09  Grafana API 요청 본문 크기 제한 미흡으로 인한 DoS 취약점이며, 구체적 영향 버전이 제
  CVE-2026-8609  CVSS 5.3   mon-graf-09  Grafana OAuth 로그인 경로의 메모리 고갈 DoS 취약점이며, 구체적 영향 버전이 제시되지
  CVE-2026-57280  CVSS 8.8   ci-jk-07     Jenkins Script Security Plugin의 샌드박스 우회 취약점으로, 플러그인 버전 
  CVE-2026-57281  CVSS 7.5   ci-jk-07     Jenkins Script Security Plugin의 샌드박스 우회 취약점으로, 플러그인 버전 
  CVE-2026-57282  CVSS 5.0   ci-jk-07     Jenkins Git client Plugin의 명령 실행 취약점으로, 

## 4단계. 위험도 평가 및 브리핑 작성

CVSS 점수는 **취약점 자체의 위험도**일 뿐, **우리 회사에서의 위험도**가 아닙니다.

| 상황 | 판단 |
|---|---|
| CVSS 7.5 + 인터넷에 노출된 결제 서버 | **긴급** |
| CVSS 9.8 + 폐쇄망 내부 개발 서버 | 보통 |

이 조정을 하는 것이 이 에이전트의 역할입니다. 그리고 결과를 **담당자별로 묶습니다.**

In [8]:
class Advisory(BaseModel):
    cve_id: str
    asset_id: str
    asset_name: str
    severity: str        # 긴급 / 높음 / 보통 / 낮음
    summary_ko: str
    action: str          # 담당자가 바로 실행할 조치
    deadline: str        # 권고 기한


class OwnerBriefing(BaseModel):
    owner: str
    owner_email: str
    headline: str        # 메일 제목
    advisories: list[Advisory]


class BriefingResult(BaseModel):
    briefings: list[OwnerBriefing]


briefer = Agent(
    name="브리핑 작성 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 보안 어드바이저리 담당자입니다. 확정된 취약점 목록과 자산 정보를 받습니다.\n"
        "위험도(severity)는 CVSS 점수만이 아니라 자산의 중요도와 인터넷 노출 여부를 함께 고려해 "
        "'긴급' / '높음' / '보통' / '낮음' 중 하나로 정하세요.\n"
        "- 인터넷에 노출되고 중요도가 '상'인 자산은 한 단계 올립니다\n"
        "- 내부망 전용이면 한 단계 내릴 수 있습니다\n"
        "action에는 담당자가 바로 실행할 수 있는 구체적 조치를 1~2문장으로 쓰세요.\n"
        "deadline은 '24시간 이내', '1주일 이내' 처럼 적으세요.\n"
        "담당자(owner)별로 묶어서 출력하고, headline은 그 담당자에게 보낼 메일 제목입니다."
    ),
    output_type=BriefingResult,
)

In [9]:
asset_info = {
    a["id"]: {
        "owner": a["owner"],
        "owner_email": a["owner_email"],
        "name": a["name"],
        "criticality": a["criticality"],
        "internet_facing": a["internet_facing"],
    }
    for a in assets
}

with trace("보안 브리핑 - 위험도 평가"):
    b = await Runner.run(
        briefer,
        json.dumps({"matched": [x.model_dump() for x in matched], "asset_info": asset_info},
                   ensure_ascii=False),
    )

briefings = b.final_output.briefings
print(f"담당자 {len(briefings)}명에게 보낼 브리핑 생성 완료\n")

for br in briefings:
    print(f"=== {br.owner} <{br.owner_email}> — {len(br.advisories)}건")
    print(f"    제목: {br.headline}")
    for a in br.advisories:
        print(f"    [{a.severity:<3}] {a.cve_id:<16} {a.asset_name:<18} {a.deadline}")
        print(f"           조치: {a.action[:80]}")
    print()

담당자 2명에게 보낼 브리핑 생성 완료

=== 김인프라 <infra.kim@example.com> — 3건
    제목: [긴급/높음] 고객포털 WAS(Tomcat) 취약점 조치 필요
    [높음 ] CVE-2026-50229   고객포털 WAS           1주일 이내
           조치: Apache Tomcat을 영향 받지 않는 최신 패치 버전으로 즉시 업그레이드하세요. 예제 애플리케이션(number guess)을 비활성화하거나
    [높음 ] CVE-2026-53404   고객포털 WAS           1주일 이내
           조치: Tomcat을 보안 패치 버전으로 업그레이드하고 rewrite valve 설정을 재점검하세요. 적용 전후로 리다이렉트/리라이트 규칙이 예상대로 
    [긴급 ] CVE-2026-55276   고객포털 WAS           24시간 이내
           조치: 서비스 영향도 확인 후 즉시 Tomcat을 패치 버전으로 업그레이드하세요. 업그레이드 전까지는 외부 노출을 최소화하고 접근 로그 및 비정상 요청

=== 최데브옵스 <devops.choi@example.com> — 6건
    제목: [확인 필요] Grafana/Jenkins 취약점 점검 및 버전 확인 요청
    [보통 ] CVE-2026-33382   모니터링 대시보드          1주일 이내
           조치: 현재 Grafana 10.1.2의 정확한 영향 여부를 공식 공지와 대조해 확인하세요. 영향 버전이면 즉시 패치하고, 그 전까지는 외부 접근을 제
    [낮음 ] CVE-2026-8609    모니터링 대시보드          1주일 이내
           조치: Grafana의 실제 영향 여부를 버전 공지로 확인하세요. 영향 시 패치하고, 로그인 엔드포인트에 대한 비정상 대량 요청을 차단하도록 WAF/프
    [높음 ] CVE-2026-57280   빌드 서버              24시간 이내
    

## 5단계. 담당자별 메일 발송 (팬아웃) — 도구로 발송

한 번의 실행에서 **담당자 수만큼 서로 다른 메일**이 나갑니다.
각자 자기 자산 건만 받으므로, 전체 목록을 첨부하는 것보다 실행률이 훨씬 높습니다.

메일 발송은 [openai_agent_code.ipynb](openai_agent_code.ipynb) Step 6 의 `send_email_tool` 처럼 **도구(function tool)** 로 만들고,
**발송 에이전트**가 이 도구를 담당자 수만큼 호출합니다.
발송 도구는 [security_briefing_by_llm.ipynb](security_briefing_by_llm.ipynb) 의 `send_briefing_email` 과 같은 방식입니다.

순서는 여전히 **코드가** 정합니다(4단계가 끝나야 5단계). 에이전트가 정하는 것은 **"누구에게 보낼지"(도구 인자)** 뿐입니다.

### 안전장치 — 수신자 라우팅

`assets.json` 의 담당자 이메일은 **가상 주소**입니다.
실수로 실제 존재하는 주소에 메일이 가지 않도록,
**모든 메일을 `.env` 의 `EMAIL_ADDRESS`(본인 주소)로 보냅니다.**
실무 적용 시 `ROUTE_ALL_TO_ME = None` 으로 바꾸면 원래 담당자에게 발송됩니다.

`DRY_RUN = True` 로 두면 메일은 보내지 않고 도구 호출 과정만 확인할 수 있습니다.

In [10]:
SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

# 데모 안전장치 1: 모든 메일을 본인 주소로 보냄. 실무에서는 None 으로 바꾼다.
ROUTE_ALL_TO_ME = EMAIL_ADDRESS

# 데모 안전장치 2: True 면 실제로 보내지 않고 "보냈다고 가정"만 한다.
DRY_RUN = False

SEVERITY_COLOR = {"긴급": "#c0392b", "높음": "#e67e22", "보통": "#2980b9", "낮음": "#7f8c8d"}


def build_html(briefing: OwnerBriefing) -> str:
    """담당자 한 명에게 보낼 HTML 메일 본문을 만듭니다."""
    rows = ""
    for a in briefing.advisories:
        color = SEVERITY_COLOR.get(a.severity, "#7f8c8d")
        rows += f"""
        <tr>
          <td style="padding:8px;border-bottom:1px solid #eee;">
            <span style="background:{color};color:#fff;padding:2px 8px;border-radius:3px;font-size:12px;">{a.severity}</span>
          </td>
          <td style="padding:8px;border-bottom:1px solid #eee;font-family:monospace;">{a.cve_id}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.asset_name}</td>
          <td style="padding:8px;border-bottom:1px solid #eee;">{a.deadline}</td>
        </tr>
        <tr>
          <td></td>
          <td colspan="3" style="padding:0 8px 12px 8px;border-bottom:1px solid #eee;color:#555;font-size:13px;">
            {a.summary_ko}<br><b>조치:</b> {a.action}
          </td>
        </tr>"""

    return f"""
    <div style="font-family:'Malgun Gothic',sans-serif;max-width:720px;">
      <h2 style="border-bottom:2px solid #333;padding-bottom:8px;">보안 취약점 일일 브리핑</h2>
      <p>{briefing.owner}님, 담당 자산에서 조치가 필요한 취약점
         <b>{len(briefing.advisories)}건</b>이 확인되었습니다.</p>
      <table style="border-collapse:collapse;width:100%;font-size:14px;">
        <tr style="background:#f4f4f4;">
          <th style="padding:8px;text-align:left;">위험도</th>
          <th style="padding:8px;text-align:left;">CVE</th>
          <th style="padding:8px;text-align:left;">자산</th>
          <th style="padding:8px;text-align:left;">조치 기한</th>
        </tr>
        {rows}
      </table>
      <p style="color:#888;font-size:12px;margin-top:16px;">
        본 메일은 보안 브리핑 에이전트가 NVD 데이터를 기반으로 자동 생성했습니다. (수업용 데모)
      </p>
    </div>"""


def send_email(to_address: str, subject: str, html_body: str) -> str:
    """HTML 메일을 발송합니다."""
    msg = EmailMessage()
    msg["Subject"] = subject
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_address
    msg.set_content("HTML 메일입니다. HTML을 지원하는 클라이언트에서 확인하세요.")
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP_SSL(SMTP_SERVER, 465) as smtp:
        smtp.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)

    return f"발송 완료 -> {to_address}"

### 발송 도구

`send_email` 을 에이전트가 부를 수 있는 **도구**로 감쌉니다.
`function_tool(함수)` 로 감싸면 SDK가 **함수 이름 · docstring · 타입 힌트**로 도구 스키마를 만듭니다.
에이전트는 코드를 보지 못하므로 **docstring 이 곧 사용 설명서**입니다.

에이전트가 넘길 수 있는 것은 **담당자 이름 하나뿐**입니다. 받는 사람 주소와 메일 본문은 코드가 정합니다.

| 안전장치 | 막아 주는 것 |
|---|---|
| 인자는 담당자 이름뿐 | LLM이 임의의 주소·본문으로 메일을 보내는 것 |
| 본문은 `build_html` 이 작성된 브리핑으로 생성 | LLM이 브리핑 내용을 지어내거나 바꾸는 것 |
| `ROUTE_ALL_TO_ME` | 데모 중 실제 담당자 주소로 나가는 것 |
| `sent_owners` 기록 | 같은 담당자에게 두 번 보내는 것 |
| `DRY_RUN` | 과정만 보고 싶을 때 메일이 실제로 나가는 것 |

In [11]:
# 발송 도구가 참조하는 데이터. 아래 "실제 발송" 셀에서 4단계 결과로 채운다.
briefings_by_owner = {}   # 담당자 -> OwnerBriefing
sent_owners = set()       # 이미 메일을 보낸 담당자 (중복 발송 방지)


def send_briefing_email(owner: str) -> str:
    """담당자 한 명에게 4단계에서 작성한 브리핑 메일을 보냅니다.
    받는 주소와 메일 본문은 작성된 브리핑으로 코드가 만들므로, 담당자 이름만 넘기면 됩니다.
    담당자 한 명당 한 번만 호출하세요.

    Args:
        owner: 담당자 이름. 입력에 나온 이름 그대로 쓰세요. (예: 김운영)
    """
    br = briefings_by_owner.get(owner)
    if br is None:
        available = ", ".join(briefings_by_owner) or "없음"
        return f"'{owner}' 의 브리핑이 없습니다. 보낼 수 있는 담당자: {available}"
    if owner in sent_owners:
        return f"{owner} 에게는 이미 발송했습니다. 다시 보내지 않습니다."

    to = ROUTE_ALL_TO_ME or br.owner_email
    if DRY_RUN:
        result = f"[DRY_RUN] 발송 생략 -> {to} / 제목: {br.headline}"
    else:
        result = send_email(to, br.headline, build_html(br))

    sent_owners.add(owner)
    return f"{owner}: {result}"


send_briefing_email_tool = function_tool(send_briefing_email)

# 에이전트(LLM)가 보는 도구의 입력 스키마
send_briefing_email_tool.params_json_schema

{'properties': {'owner': {'description': '담당자 이름. 입력에 나온 이름 그대로 쓰세요. (예: 김운영)',
   'title': 'Owner',
   'type': 'string'}},
 'required': ['owner'],
 'title': 'send_briefing_email_args',
 'type': 'object',
 'additionalProperties': False}

### 발송 에이전트

[openai_agent_code.ipynb](openai_agent_code.ipynb) Step 7 의 `sales_sender` 와 같은 구조입니다.

`ModelSettings(tool_choice="required")` 는 에이전트가 **첫 응답에서 반드시 도구를 호출**하게 합니다.
도구를 한 번 호출하고 나면 SDK가 자동으로 `auto` 로 되돌리므로(`reset_tool_choice=True` 기본값),
도구만 계속 부르는 무한 루프에 빠지지 않습니다.

In [ ]:
sender = Agent(
    name="메일 발송 에이전트",
    model=MODEL_NAME,
    instructions=(
        "당신은 보안 브리핑 발송 담당자입니다. 담당자별 브리핑 요약 목록을 받습니다.\n"
        "목록에 있는 담당자마다 send_briefing_email 도구를 한 번씩 호출하세요.\n"
        "메일 주소와 본문은 도구가 만들므로 담당자 이름만 넘기면 됩니다.\n"
        "모두 끝나면 담당자별 발송 결과를 한 줄씩 보고하세요."
    ),
    tools=[send_briefing_email_tool],
    model_settings=ModelSettings(tool_choice="required"),
)

### 발송 전 미리보기

**외부로 나가는 행동은 반드시 눈으로 확인하고 실행합니다.**
먼저 무엇이 누구에게 갈지 출력만 해 봅니다.

In [11]:
for br in briefings:
    to = ROUTE_ALL_TO_ME or br.owner_email
    print(f"받는사람: {to}   (원래 담당자: {br.owner} <{br.owner_email}>)")
    print(f"제목    : {br.headline}")
    print(f"내용    : {len(br.advisories)}건 / "
          f"{', '.join(a.severity for a in br.advisories)}")
    print("-" * 70)

받는사람: sanchoiree@gmail.com   (원래 담당자: 김운영 <kim.operations@hanbit-tech.example>)
제목    : 긴급: 대외 노출 Apache 취약점 패치 필요 (SRV-WEB-01, SRV-WAS-01)
내용    : 5건 / 긴급, 높음, 긴급, 높음, 긴급
----------------------------------------------------------------------
받는사람: sanchoiree@gmail.com   (원래 담당자: 이개발 <lee.dev@hanbit-tech.example>)
제목    : 패치 필요: Spring Framework 취약점 점검 (APP-BE-01)
내용    : 3건 / 보통, 보통, 보통
----------------------------------------------------------------------


### 실제 발송

위 내용을 확인했다면 아래 셀을 실행합니다. **발송 에이전트가 도구를 호출해 실제로 메일이 나갑니다.**
(과정만 보려면 먼저 `DRY_RUN = True` 로 바꾸세요.)

- 에이전트에게는 메일 본문이 아니라 **누구에게 무엇이 갈지 요약만** 넘깁니다. 본문은 도구가 `build_html` 로 만듭니다.
- 결과는 에이전트의 보고가 아니라 **도구가 실제로 돌려준 값**으로 확인합니다.
- 마지막으로 빠진 담당자가 없는지 **코드가** 확인합니다. 순서와 검증은 코드의 몫입니다.

In [ ]:
# 발송 도구가 참조할 데이터를 4단계 결과로 채운다 (이 셀을 다시 실행하면 새로 발송)
briefings_by_owner = {br.owner: br for br in briefings}
sent_owners = set()

# 발송 에이전트에게는 메일 본문이 아니라 누구에게 무엇이 갈지 요약만 넘긴다
sender_input = "\n".join(
    f"- {br.owner}: {len(br.advisories)}건 / 제목: {br.headline}" for br in briefings
)

with trace("보안 브리핑 - 메일 발송"):
    s = await Runner.run(sender, sender_input)

# 에이전트의 보고가 아니라 도구가 실제로 돌려준 결과를 출력한다
for item in s.new_items:
    if item.type == "tool_call_output_item":
        print(item.output)

# 코드 오케스트레이션: 빠진 담당자가 없는지 코드가 확인한다
missing = sorted(set(briefings_by_owner) - sent_owners)
print(f"\n발송 {len(sent_owners)}/{len(briefings_by_owner)}명 — "
      + (f"누락: {', '.join(missing)}" if missing else "누락 없음"))

## 정리

### 이 예제에서 확인한 것

| 단계 | 확인한 내용 |
|---|---|
| 2단계 | 실제 외부 API(NVD)를 도구로 연결 — 데이터를 지어내지 않음 |
| 3단계 | 에이전트가 **키워드만 겹치는 오탐을 걸러냄** — 규칙으로는 불가능한 판단 |
| 4단계 | CVSS 점수를 **자사 맥락(중요도·노출)으로 재해석** |
| 5단계 | 발송 에이전트가 메일 도구로 **담당자별 개인화 메일 발송** (팬아웃). 주소·본문·중복 방지는 도구 코드가 통제 |

### 실무 적용 시 추가로 필요한 것

1. **자산 정보를 CMDB와 연동** — `assets.json` 대신 실제 자산관리시스템 조회
2. **스케줄 실행** — 매일 아침 8시 자동 실행 (Windows 작업 스케줄러 / cron / GitHub Actions)
3. **중복 발송 방지** — 이미 통보한 CVE는 기록해 두고 제외
4. **조치 결과 추적** — 담당자가 패치했는지 확인하는 후속 흐름
5. **`trace` 결과 보관** — "왜 이 취약점을 긴급으로 분류했는가"에 대한 감사 대응

### 비용

이 파이프라인 1회 실행은 에이전트 호출 3번(매칭, 브리핑, 발송)입니다.
매일 1회 실행해도 부담이 크지 않으며, 자산이 늘어나면 후보 CVE 수에 비례해 증가합니다.
비용을 더 줄이려면 3단계 매칭처럼 단순한 판단은 더 작은 모델로 내리면 됩니다.
자세한 내용은 [context_token_num.md](../contextw/context_token_num.md) 를 참고하세요.

### 다음 단계

같은 부품으로 **순서까지 LLM이 정하는** 버전은 [security_briefing_by_llm.ipynb](security_briefing_by_llm.ipynb) 를 참고하세요.